In [3]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from collections import Counter

In [4]:

class BaggingClassifierScratch:
    """
    Bagging (Bootstrap Aggregating) ensemble classifier, built from scratch.

    Trains `n_estimators` independent decision trees, each on a bootstrap
    resample (sampling with replacement) of the training data, then combines
    their predictions via majority vote. Reduces variance vs a single tree
    without needing each tree to be individually well-tuned.
    """

    def __init__(self, n_estimators=10, max_depth=None, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state

        self.models = []
        self.oob_score_ = None

    def _bootstrap_sample(self, X, y, rng):
        n_samples = X.shape[0]
        indices = rng.choice(n_samples, size=n_samples, replace=True)
        # unique indices NOT selected -> out-of-bag for this tree
        oob_mask = np.ones(n_samples, dtype=bool)
        oob_mask[np.unique(indices)] = False
        return X[indices], y[indices], oob_mask

    def fit(self, X, y):
        self.models = []
        n_samples = X.shape[0]

        rng = np.random.default_rng(self.random_state)

        # track OOB predictions per sample across all trees that didn't see it
        oob_votes = [[] for _ in range(n_samples)]

        for i in range(self.n_estimators):
            X_sample, y_sample, oob_mask = self._bootstrap_sample(X, y, rng)

            model = DecisionTreeClassifier(
                max_depth=self.max_depth,
                random_state=None if self.random_state is None else self.random_state + i
            )
            model.fit(X_sample, y_sample)
            self.models.append(model)

            # collect OOB predictions for this tree
            oob_indices = np.where(oob_mask)[0]
            if len(oob_indices) > 0:
                oob_preds = model.predict(X[oob_indices])
                for idx, p in zip(oob_indices, oob_preds):
                    oob_votes[idx].append(p)

        # compute OOB score across samples that had at least one OOB vote
        oob_preds_final = []
        oob_true = []
        for idx, votes in enumerate(oob_votes):
            if len(votes) > 0:
                oob_preds_final.append(Counter(votes).most_common(1)[0][0])
                oob_true.append(y[idx])

        if len(oob_true) > 0:
            self.oob_score_ = accuracy_score(oob_true, oob_preds_final)
        else:
            self.oob_score_ = None  # not enough estimators to get OOB coverage

        return self

    def predict(self, X):
        predictions = np.array([model.predict(X) for model in self.models])
        return np.array([
            Counter(sample_predictions).most_common(1)[0][0]
            for sample_predictions in predictions.T
        ])

    def predict_proba(self, X):
        """Fraction of trees voting for each class, per sample."""
        predictions = np.array([model.predict(X) for model in self.models])
        classes = np.unique(predictions)
        proba = np.zeros((X.shape[0], len(classes)))

        for i, sample_predictions in enumerate(predictions.T):
            counts = Counter(sample_predictions)
            for j, c in enumerate(classes):
                proba[i, j] = counts.get(c, 0) / self.n_estimators

        return proba

In [5]:
# Demo / validation against sklearn
if __name__ == "__main__":
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import BaggingClassifier
    from sklearn.tree import DecisionTreeClassifier as DTC

    X, y = make_classification(
        n_samples=500, n_features=10, n_informative=6,
        n_redundant=2, random_state=42
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # baseline: single tree
    single = DTC(random_state=42)
    single.fit(X_train, y_train)
    print("Single Tree Test Acc:      ", accuracy_score(y_test, single.predict(X_test)))

    # scratch bagging
    bag = BaggingClassifierScratch(n_estimators=50, random_state=42)
    bag.fit(X_train, y_train)
    pred = bag.predict(X_test)
    print("Scratch Bagging Test Acc:  ", accuracy_score(y_test, pred))
    print("Scratch Bagging OOB Acc:   ", bag.oob_score_)

    # sklearn bagging, for cross-validation of the scratch version
    skb = BaggingClassifier(estimator=DTC(), n_estimators=50,
                             oob_score=True, random_state=42)
    skb.fit(X_train, y_train)
    print("sklearn Bagging Test Acc:  ", accuracy_score(y_test, skb.predict(X_test)))
    print("sklearn Bagging OOB Acc:   ", skb.oob_score_)

Single Tree Test Acc:       0.82
Scratch Bagging Test Acc:   0.88
Scratch Bagging OOB Acc:    0.875
sklearn Bagging Test Acc:   0.89
sklearn Bagging OOB Acc:    0.8625
